# Entrenamiento del modelo general (`shelf-generic-yolo26m`)

Detecta dos clases en fotos de góndolas: **`product`** y **`gap`** (espacio vacío).

## 1. Rutas

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent

DATA_YAML = ROOT / "data/datasets/shelf-generic/data.yaml"
PRETRAINED = ROOT / "models/pretrained/yolo26m.pt"
MODEL_DIR = ROOT / "models/shelf-generic-yolo26m"
RUNS_DIR = ROOT / "runs/shelf-generic-yolo26m"

## 2. Verificar la GPU

In [ ]:
import torch

print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 3. Entrenar

Parte de `yolo26m` preentrenado en COCO. Solo se cambian estos valores; el resto son los defaults de Ultralytics:

| parámetro | valor | por qué |
|---|---|---|
| `imgsz` | 1024 | los productos son pequeños en la foto completa |
| `epochs` / `patience` | 60 / 15 | corta si no mejora en 15 épocas |
| `batch` | 8 | cabe en 16 GB de VRAM a 1024 px |
| `degrees` / `perspective` | 5 / 0.0005 | fotos tomadas con el celular un poco inclinadas |

In [ ]:
from ultralytics import YOLO

model = YOLO(PRETRAINED)
model.train(
    data=DATA_YAML,
    imgsz=1024,
    epochs=60,
    patience=15,
    batch=8,
    degrees=5.0,
    perspective=0.0005,
    project=RUNS_DIR,
    name="train",
    exist_ok=True,
)

## 4. Guardar el mejor modelo

In [ ]:
import shutil

shutil.copy(RUNS_DIR / "train/weights/best.pt", MODEL_DIR / "best.pt")

## 5. Evaluar en test

In [ ]:
metrics = YOLO(MODEL_DIR / "best.pt").val(data=DATA_YAML, split="test", imgsz=1024, project=RUNS_DIR, name="test-1024", exist_ok=True)